# 🚀 E-Commerce Delivery Delay - CatBoost Cloud Training

This notebook is optimized for **Google Colab GPUs** to perform hyperparameter tuning on the full dataset.
CatBoost has been identified as the superior model for this task.

### **Instructions:**
1. Ensure you have selected a **GPU runtime** (Runtime -> Change runtime type -> T4 GPU).
2. **Run the first cell** to upload your `features.csv` file.
3. Run all cells to tune and download the final model.

In [ ]:
# --- 1. UPLOAD DATA ---
import ipywidgets as widgets
from IPython.display import display
import os

uploader = widgets.FileUpload(
    accept='.csv', 
    multiple=False,
    description='Select features.csv'
)
output = widgets.Output()

def on_upload_change(change):
    with output:
        output.clear_output()
        if not uploader.value: return
        uploaded_file = uploader.value[0]
        content = uploaded_file['content']
        with open("features.csv", "wb") as f:
            f.write(content)
        print(f"✅ SUCCESS: Saved as 'features.csv' ({len(content)/1024/1024:.2f} MB)")

uploader.observe(on_upload_change, names='value')
display(uploader, output)

In [ ]:
# --- 2. INSTALL LIBRARIES ---
%pip install catboost optuna pandas loguru

In [ ]:
import pandas as pd
import numpy as np
import optuna
import json
import os
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score

# Settings
DATA_PATH = "features.csv"
N_TRIALS = 50
N_SPLITS = 3
TARGET = 'is_late'

if not os.path.exists(DATA_PATH):
    print("❌ ERROR: features.csv not found. Please upload it first!")
else:
    df = pd.read_csv(DATA_PATH)
    X = df.drop(columns=['order_id', TARGET])
    y = df[TARGET].astype(int)

    cat_cols = ['customer_state', 'seller_state', 'product_category', 
                'primary_payment_type', 'purchase_month', 'purchase_day_of_week', 'purchase_hour']
    
    # Standardize types for CatBoost
    for col in cat_cols:
        if col in X.columns:
            X[col] = X[col].fillna("UNKNOWN").astype(str)

    print(f"Loaded {len(df)} rows and {len(X.columns)} features. Ready to optimize!")

In [ ]:
# --- 3. CATBOOST GPU HYPERPARAMETER TUNING ---
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 2000),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.01, 10, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.01, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1),
        "task_type": "GPU",
        "auto_class_weights": "Balanced",
        "verbose": False,
        "random_seed": 42
    }
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    scores = []
    
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, cat_features=cat_cols)
        
        y_prob = model.predict_proba(X_val)[:, 1]
        scores.append(average_precision_score(y_val, y_prob))
    
    return np.mean(scores)

print("Starting 50 Trials of CatBoost optimization on GPU...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS)

print(f"\n🏆 Best CatBoost PR-AUC: {study.best_value:.4f}")
print(f"Best Params: {study.best_params}")

In [ ]:
# --- 4. TRAIN FINAL MODEL & DOWNLOAD ---
print("Training final model on full dataset with best parameters...")
final_model = CatBoostClassifier(
    **study.best_params, 
    task_type="GPU", 
    auto_class_weights="Balanced", 
    verbose=100,
    random_seed=42
)
final_model.fit(X, y, cat_features=cat_cols)

# Save artifacts
final_model.save_model("catboost_tuned.cbm")
with open("best_catboost_params.json", "w") as f: 
    json.dump(study.best_params, f, indent=4)

# Feature Importance
fi = final_model.get_feature_importance()
fi_df = pd.DataFrame({'feature': X.columns, 'importance': fi}).sort_values('importance', ascending=False)
fi_df.to_csv("catboost_feature_importance.csv", index=False)
print("\nTop 10 Features:")
print(fi_df.head(10))

print("\n✅ All artifacts saved. Downloading results...")
from google.colab import files
import subprocess

subprocess.run(["zip", "cloud_results.zip", "catboost_tuned.cbm", "best_catboost_params.json", "catboost_feature_importance.csv"])
files.download("cloud_results.zip")